In [55]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
RESPOND news — merge sessions + sample → meta-frames → descriptives

What this script does
---------------------
1) Load & flatten coder session JSONs (one row per article/annotation).
2) Load the original manual-coding sample (merged CSV or per-country CSVs).
3) Merge sample + sessions (auto-select join key).
4) Recode frame variables to 0/1 and build 3 meta-frames.
5) Create date levels from `dateTime`: year, month, year_month, iso_yearweek.
6) Print general descriptives (overall, by country/outlet/month, date ranges, missingness).
7) Save a single merged CSV with all variables + a few small summary CSVs.

NOTE: Adjust the config paths below if needed.
"""

import os
import json
from datetime import datetime
from typing import Dict, Any, Iterable, List, Optional, Tuple

import pandas as pd


# =========================
# 0) CONFIG — edit paths if needed
# =========================
SESSIONS_PATH = os.path.expanduser(
    "~/webdav/ASCOR-FMG-5580-RESPOND-news-data (Projectfolder)/annotations/coding_frames/final_sample/sessions/"
)
SAMPLE_PATH = os.path.expanduser(
    "~/webdav/ASCOR-FMG-5580-RESPOND-news-data (Projectfolder)/output/data-deductive-analysis/sample-manual-content-analysis"
)

SESSION_NAMES = [
    "Alexander_session.json",
    "Assia_session.json",
    "Luigia_session",        # also tries Luigia_session.json
    "Elisa_session",         # also tries Elisa_session.json
]

ANNOTATOR_MAP = {
    "Bulgaria": "Alexander",
    "Italy": "Luigia",
    "Netherlands": "Assia",
    "United_Kingdom": "Elisa",
}

# If files use other keys, add them (earlier = higher priority)
CANDIDATE_JOIN_KEYS = [
    ["uri"],
    ["url"],
    ["article_url"],
    ["link"],
    ["guid"],
    ["article_id"],
    ["id"],
    ["article.index"],
    ["article_id_numeric"],
    ["article_index"],
    ["country", "uri"],  # composite fallbacks
    ["country", "url"],
]

# Frame columns expected in merged data
FRAME_COLS = [
    "Foreign influence threat_present",
    "Systemic institutional corruption_present",
    "Elite collusion_present",
    "Politicized investigations_present",
    "Authoritarian reformism_present",
    "Judicial and institutional accountability failures_present",
    "Mobilizing anti-corruption_present",
]

# The date column present in your merged data
DATE_COL = "dateTime"   # IMPORTANT: exists in your data


# =========================
# 1) Session helpers
# =========================
def find_existing_session_paths(base_dir: str, names: Iterable[str]) -> Dict[str, str]:
    """Return actual paths that exist on disk for the requested session names."""
    results = {}
    for name in names:
        candidates = [name] if name.endswith(".json") else [name, f"{name}.json"]
        for cand in candidates:
            p = os.path.join(base_dir, cand)
            if os.path.isfile(p):
                results[name] = p
                break
    return results


def load_json(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_session(session_json: Dict[str, Any], source_file: str) -> pd.DataFrame:
    """
    Flatten a session to 1 row per annotation/article.
    - Keeps all annotation variables (dot-flatten nested).
    - Adds session-level fields prefixed with session_.
    """
    session_fields = {f"session_{k}": v for k, v in session_json.items() if k != "annotations"}
    session_fields["source_file"] = os.path.basename(source_file)

    anns = session_json.get("annotations", [])
    if not isinstance(anns, list) or len(anns) == 0:
        return pd.DataFrame([{**session_fields}])

    ann_df = pd.json_normalize(anns, sep=".")
    for k, v in session_fields.items():
        ann_df[k] = v

    session_cols = list(session_fields.keys())
    other_cols = [c for c in ann_df.columns if c not in session_cols]
    return ann_df[session_cols + other_cols]


def build_annotations_df() -> Tuple[pd.DataFrame, List[str], List[str]]:
    """Load and flatten all session JSONs into one DataFrame."""
    existing = find_existing_session_paths(SESSIONS_PATH, SESSION_NAMES)
    missing = []
    for name in SESSION_NAMES:
        if name not in existing and f"{name}.json" not in existing:
            missing.append(name if name.endswith(".json") else f"{name}.json")

    frames = []
    for _, path in existing.items():
        data = load_json(path)
        frames.append(normalize_session(data, source_file=path))

    if not frames:
        return pd.DataFrame(), list(existing.values()), missing

    df = pd.concat(frames, ignore_index=True, sort=False)

    # Optional annotator inference
    if "session_user_id" in df.columns and df["session_user_id"].notna().any():
        df["annotator_inferred"] = df["session_user_id"]
    else:
        df["annotator_inferred"] = df["source_file"].str.extract(r"^(.*?)_", expand=False)

    return df, list(existing.values()), missing


# =========================
# 2) Load original sample
# =========================
def load_original_sample() -> pd.DataFrame:
    """
    Tries:
      1) dataset_for_manual_content_analysis_250_per_country.csv
      2) concatenation of per-country CSVs (e.g., Bulgaria_Alexander_sample_250.csv).
    """
    merged_csv = os.path.join(SAMPLE_PATH, "dataset_for_manual_content_analysis_250_per_country.csv")
    if os.path.isfile(merged_csv):
        return pd.read_csv(merged_csv)

    pieces = []
    for country, annot in ANNOTATOR_MAP.items():
        fn = f"{country}_{annot}_sample_250.csv"
        path = os.path.join(SAMPLE_PATH, fn)
        if os.path.isfile(path):
            pieces.append(pd.read_csv(path))
    if pieces:
        return pd.concat(pieces, ignore_index=True, sort=False)

    raise FileNotFoundError(
        "Could not find the original sample CSV. Expected either:\n"
        f"  - {merged_csv}\n"
        "  - Or the per-country CSVs in the same folder."
    )


# =========================
# 3) Merge logic
# =========================
def choose_best_join_keys(left: pd.DataFrame, right: pd.DataFrame, candidates: List[List[str]]) -> Optional[List[str]]:
    """Pick the first candidate key set that exists in both frames and produces any matches."""
    left_cols = set(map(str, left.columns))
    right_cols = set(map(str, right.columns))
    for keys in candidates:
        if all(k in left_cols for k in keys) and all(k in right_cols for k in keys):
            lsub = left.dropna(subset=keys)
            rsub = right.dropna(subset=keys)
            if len(lsub) == 0 or len(rsub) == 0:
                continue
            tmp = lsub.merge(rsub[keys].drop_duplicates(), on=keys, how="inner")
            if len(tmp) > 0:
                return keys
    return None


def merge_all_variables(sample_df: pd.DataFrame, ann_df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Merge sample + annotations while preserving all columns; return merged and a report dict."""
    sample_df = sample_df.copy()
    if "country" in sample_df.columns and "annotator" not in sample_df.columns:
        sample_df["annotator_expected"] = sample_df["country"].map(ANNOTATOR_MAP).fillna("")

    keys = choose_best_join_keys(sample_df, ann_df, CANDIDATE_JOIN_KEYS)

    report: Dict[str, Any] = {}
    if keys is None:
        report["join_keys"] = None
        report["note"] = "No suitable join key found. Update CANDIDATE_JOIN_KEYS."
        return sample_df, report

    merged = sample_df.merge(ann_df, on=keys, how="left", suffixes=("", "_ann"), copy=False)

    only_in_sample = sample_df.merge(ann_df[keys].drop_duplicates(), on=keys, how="left", indicator=True)
    only_in_sample = only_in_sample[only_in_sample["_merge"] == "left_only"][keys]

    only_in_ann = ann_df.merge(sample_df[keys].drop_duplicates(), on=keys, how="left", indicator=True)
    only_in_ann = only_in_ann[only_in_ann["_merge"] == "left_only"][keys]

    report.update(
        dict(
            join_keys=keys,
            sample_rows=len(sample_df),
            ann_rows=len(ann_df),
            merged_rows=len(merged),
            unmatched_in_annotations=len(only_in_sample),
            annotations_without_sample=len(only_in_ann),
        )
    )
    return merged, report


# =========================
# 4) Frames → 0/1 and meta-frames
# =========================
def recode_present(series: pd.Series) -> pd.Series:
    """Recode textual present/absent (and common variants) to 0/1 nullable Int."""
    s = series.astype(str).str.strip().str.lower()
    mapped = s.replace({
        "present": 1,
        "not present": 0,
        "yes": 1, "no": 0,
        "true": 1, "false": 0,
        "1": 1, "0": 0,
        "nan": pd.NA, "": pd.NA
    })
    return pd.to_numeric(mapped, errors="coerce").astype("Int64")


def add_meta_frames(df: pd.DataFrame) -> None:
    """Add 3 meta-frame columns (binary; NA if all constituents NA) directly to df."""
    # Step 1: recode the seven frames to 0/1
    for col in FRAME_COLS:
        if col in df.columns:
            df[col] = recode_present(df[col])

    # Step 2: aggregate into meta-frames
    sysfail_cols = [
        "Systemic institutional corruption_present",
        "Judicial and institutional accountability failures_present",
    ]
    eliteabuse_cols = [
        "Elite collusion_present",
        "Politicized investigations_present",
        "Authoritarian reformism_present",
        "Foreign influence threat_present",
    ]
    account_cols = ["Mobilizing anti-corruption_present"]

    def any_present(cols: List[str], out_col: str):
        existing = [c for c in cols if c in df.columns]
        if not existing:
            df[out_col] = pd.NA
            return
        sub = df[existing].copy()
        df[out_col] = sub.max(axis=1, skipna=True)
        all_na = sub.isna().all(axis=1)
        df.loc[all_na, out_col] = pd.NA
        df[out_col] = pd.to_numeric(df[out_col], errors="coerce").astype("Int64")

    any_present(sysfail_cols,  "meta1_system_failure")
    any_present(eliteabuse_cols,"meta2_elite_abuse")
    any_present(account_cols,  "meta3_accountability")


# =========================
# 5) Date levels from `dateTime`
# =========================
def add_date_levels(df: pd.DataFrame, date_col: str = DATE_COL) -> None:
    """Create year, month, year_month, iso_yearweek from the given date column."""
    if date_col not in df.columns:
        return
    dt = pd.to_datetime(df[date_col], errors="coerce", utc=True).dt.tz_localize(None)
    df["_dt"] = dt
    df["year"] = df["_dt"].dt.year.astype("Int64")
    df["month"] = df["_dt"].dt.month.astype("Int64")
    df["year_month"] = df["_dt"].dt.to_period("M").astype(str)
    df["iso_yearweek"] = df["_dt"].dt.strftime("%G-W%V")


# =========================
# 6) Descriptives
# =========================
def print_and_save_descriptives(df: pd.DataFrame, out_dir: str) -> None:
    """Print and save basic descriptives: coverage, prevalence, by country/outlet/month, date range."""
    def safe_pct(num, den):
        return (100 * num / den) if den else 0.0

    # Annotation coverage (heuristic: any session_* or source_file present)
    ann_like_cols = [c for c in df.columns if c.startswith("session_") or c == "source_file" or c.endswith("_ann")]
    if ann_like_cols:
        df["is_annotated"] = df[ann_like_cols].notna().any(axis=1)
    else:
        df["is_annotated"] = True

    n_total = len(df)
    n_annot = int(df["is_annotated"].sum())
    print("\n=== Overall ===")
    print(f"Rows: {n_total}  |  Annotated: {n_annot} ({safe_pct(n_annot, n_total):.1f}%)")

    # Date range
    if "_dt" in df.columns:
        print("\n=== Date range ===")
        dtmin = df["_dt"].min()
        dtmax = df["_dt"].max()
        print(f"From: {dtmin}  To: {dtmax}")

    # Meta prevalence overall (excluding NA)
    metas = ["meta1_system_failure","meta2_elite_abuse","meta3_accountability"]
    print("\n=== Meta-frame prevalence (overall) ===")
    for m in metas:
        if m in df.columns:
            s = df[m].dropna()
            present = int((s == 1).sum())
            denom = len(s)
            print(f"{m}: 1={present}/{denom}  ({safe_pct(present, denom):.1f}%)")

    # By country
    if "country" in df.columns:
        print("\n=== By country (counts) ===")
        country_counts = df["country"].value_counts(dropna=False).sort_index()
        print(country_counts)

        print("\n=== Meta prevalence by country (row = country, values = % present) ===")
        by_country = (
            df.dropna(subset=metas)
              .groupby("country")[metas]
              .mean()
              .mul(100)
              .round(1)
              .sort_index()
        )
        print(by_country)
        by_country.to_csv(os.path.join(out_dir, "summary_meta_by_country.csv"), index=True)

    # By outlet (top 15 by volume)
    outlet_col = None
    for cand in ["source.uri", "source_uri", "source", "publisher"]:
        if cand in df.columns:
            outlet_col = cand
            break
    if outlet_col:
        print("\n=== Top outlets by volume ===")
        top_outlets = df[outlet_col].value_counts().head(15)
        print(top_outlets)
        top_outlets.to_csv(os.path.join(out_dir, "summary_top_outlets.csv"), header=["n"])

        print("\n=== Meta prevalence by outlet (top 15 by volume) ===")
        top_names = top_outlets.index.tolist()
        by_outlet = (
            df[df[outlet_col].isin(top_names)]
            .dropna(subset=metas)
            .groupby(outlet_col)[metas]
            .mean()
            .mul(100)
            .round(1)
            .loc[top_names]  # keep same order
        )
        print(by_outlet)
        by_outlet.to_csv(os.path.join(out_dir, "summary_meta_by_top_outlets.csv"), index=True)

    # By month (volume + prevalence)
    if "year_month" in df.columns:
        print("\n=== Monthly volume (first 12 shown) ===")
        month_vol = df.groupby("year_month").size().rename("n").reset_index()
        print(month_vol.head(12))
        month_vol.to_csv(os.path.join(out_dir, "summary_monthly_volume.csv"), index=False)

        print("\n=== Monthly meta prevalence (first 12 rows shown) ===")
        month_prev = (
            df.dropna(subset=metas)
              .groupby("year_month")[metas]
              .mean()
              .mul(100)
              .round(1)
              .reset_index()
        )
        print(month_prev.head(12))
        month_prev.to_csv(os.path.join(out_dir, "summary_monthly_meta.csv"), index=False)

    # Missingness quick view
    print("\n=== Missingness (selected columns) ===")
    sel = ["country", outlet_col if outlet_col else None, DATE_COL,
           "year", "month", "year_month",
           "meta1_system_failure","meta2_elite_abuse","meta3_accountability"]
    sel = [c for c in sel if c and c in df.columns]
    miss = df[sel].isna().mean().mul(100).round(1).rename("pct_missing").to_frame()
    print(miss)
    miss.to_csv(os.path.join(out_dir, "summary_missingness.csv"))


# =========================
# 7) MAIN
# =========================
def main() -> None:
    # Load/flatten sessions
    ann_df, found_paths, missing_files = build_annotations_df()
    print("=== Sessions loaded ===")
    print("Found:", ", ".join(os.path.basename(p) for p in found_paths) if found_paths else "(none)")
    if missing_files:
        print("Missing:", ", ".join(missing_files))
    print(f"Annotation rows: {len(ann_df)}  |  cols: {len(ann_df.columns)}")

    # Load original sample
    sample_df = load_original_sample()
    print("\n=== Original sample loaded ===")
    print(f"Sample rows: {len(sample_df)}  |  cols: {len(sample_df.columns)}")

    # Merge
    merged_df, report = merge_all_variables(sample_df, ann_df)
    print("\n=== Merge report ===")
    for k, v in report.items():
        print(f"{k}: {v}")

    # Add frames + meta-frames
    add_meta_frames(merged_df)

    # Add date levels from `dateTime`
    add_date_levels(merged_df, DATE_COL)

    # Convenience: also expose a dot-free outlet column (keeps originals too)
    if "source.uri" in merged_df.columns and "source_uri" not in merged_df.columns:
        merged_df["source_uri"] = merged_df["source.uri"].astype(str)

    # Save single, full CSV (everything in one place)
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    out_csv = os.path.join(SAMPLE_PATH, f"manual_sample_PLUS_annotations_{ts}.csv")
    merged_df.to_csv(out_csv, index=False, encoding="utf-8-sig")

    print("\n=== Output ===")
    print("Merged CSV (single file with all variables):", out_csv)

    # Descriptives (printed and saved as small CSVs next to the merged file)
    print_and_save_descriptives(merged_df, out_dir=SAMPLE_PATH)

    # Optional preview
    preview_cols = [c for c in [
        "country", "uri", "source.uri", "source_uri", "year_month",
        "meta1_system_failure", "meta2_elite_abuse", "meta3_accountability"
    ] if c in merged_df.columns]
    print("\n=== Preview ===")
    print(merged_df[preview_cols].head(10))


if __name__ == "__main__":
    main()


=== Sessions loaded ===
Found: Alexander_session.json, Assia_session.json, Luigia_session.json, Elisa_session.json
Annotation rows: 872  |  cols: 20

=== Original sample loaded ===
Sample rows: 1000  |  cols: 10

=== Merge report ===
join_keys: ['uri']
sample_rows: 1000
ann_rows: 872
merged_rows: 1000
unmatched_in_annotations: 128
annotations_without_sample: 0

=== Output ===
Merged CSV (single file with all variables): /home/akroon/webdav/ASCOR-FMG-5580-RESPOND-news-data (Projectfolder)/output/data-deductive-analysis/sample-manual-content-analysis/manual_sample_PLUS_annotations_20250912-101054.csv

=== Overall ===
Rows: 1000  |  Annotated: 872 (87.2%)

=== Date range ===
From: 2018-01-01 16:41:00  To: 2024-12-23 18:08:44

=== Meta-frame prevalence (overall) ===
meta1_system_failure: 1=350/872  (40.1%)
meta2_elite_abuse: 1=523/872  (60.0%)
meta3_accountability: 1=194/872  (22.2%)

=== By country (counts) ===
country
Bulgaria          250
Italy             250
Netherlands       250
Unit